# Debate LLM App

This notebook demonstrates how to build a simple Debate App using an LLM and Gradio.

We will use:
- `gradio` for the User Interface.
- `openrouter` API for the debate logic (requires API Key).

In [2]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

# Initialization

load_dotenv(override=True)

if not os.getenv("OPENROUTER_API_KEY"):
    print("Set OPENROUTER_API_KEY in .env (repo root or this folder).")
else:
    print("OPENROUTER_API_KEY loaded.")

client = OpenAI(base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),)

/Users/mubaraq/Documents/Python/AI Projects/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OPENROUTER_API_KEY loaded.


## 1. Define the Debate Logic

This function handles the core logic. It uses the OpenAI API if a key is provided; otherwise, it falls back to a mock response for testing.

In [7]:
def run_debate(topic, rounds=3, model_pro="google/gemini-2.0-flash-001", model_con="google/gemini-2.0-flash-001"):
    """
    Runs a debate between two AI agents: Pro vs Con.
    Yields the chat history for Gradio to stream.
    """
    if not client:
        yield [{"role": "assistant", "content": "Error: OPENROUTER_API_KEY not found."}]
        return

    # System Prompts
    system_pro = f"You are a skilled debater arguing FOR the topic: '{topic}'. Keep your arguments sharp, concise (max 2-3 sentences), and persuasive."
    system_con = f"You are a skilled debater arguing AGAINST the topic: '{topic}'. Keep your arguments sharp, concise (max 2-3 sentences), and persuasive."

    messages_pro = [{"role": "system", "content": system_pro}]
    messages_con = [{"role": "system", "content": system_con}]
    
    history = []
    
    # --- Round 1: Opening ---
    # Pro starts
    messages_pro.append({"role": "user", "content": "Present your opening argument."})
    
    try:
        pro_msg = client.chat.completions.create(model=model_pro, messages=messages_pro).choices[0].message.content
        messages_pro.append({"role": "assistant", "content": pro_msg})
        
        # Add Pro's message to history (User side)
        history.append({"role": "user", "content": pro_msg})
        yield history

        # Con responds
        messages_con.append({"role": "user", "content": f"The opponent argues: \"{pro_msg}\". Present your counter-argument."})
        con_msg = client.chat.completions.create(model=model_con, messages=messages_con).choices[0].message.content
        messages_con.append({"role": "assistant", "content": con_msg})
        
        # Add Con's message to history (Assistant side)
        history.append({"role": "assistant", "content": con_msg})
        yield history
        
    except Exception as e:
        history.append({"role": "assistant", "content": f"Error: {str(e)}"})
        yield history
        return

    # --- Subsequent Rounds ---
    last_con_msg = con_msg
    for _ in range(rounds - 1):
        try:
            # Pro responds to Con
            messages_pro.append({"role": "user", "content": f"The opponent argues: \"{last_con_msg}\". Rebut this and advance your case."})
            pro_msg = client.chat.completions.create(model=model_pro, messages=messages_pro).choices[0].message.content
            messages_pro.append({"role": "assistant", "content": pro_msg})
            
            # Show Pro's message immediately
            history.append({"role": "user", "content": pro_msg})
            yield history
            
            # Con responds to Pro
            messages_con.append({"role": "user", "content": f"The opponent argues: \"{pro_msg}\". Rebut this and advance your case."})
            con_msg = client.chat.completions.create(model=model_con, messages=messages_con).choices[0].message.content
            messages_con.append({"role": "assistant", "content": con_msg})
            last_con_msg = con_msg
            
            # Update the last entry with Con's response
            history.append({"role": "assistant", "content": con_msg})
            yield history
            
        except Exception as e:
            history.append({"role": "assistant", "content": f"Error: {str(e)}"})
            yield history
            break

## 2. Build the Gradio UI

We will create a UI that allows the user to:
1.  Enter a **Topic** for the debate.
2.  Choose a **Stance** (Pro or Con) for the AI.
3.  Chat with the AI.

In [10]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🗣️ AI vs AI Debate Arena")
    gr.Markdown("Watch two AI agents debate any topic!")
    
    with gr.Row():
        with gr.Column(scale=1):
            topic_input = gr.Textbox(label="Debate Topic", placeholder="e.g., Universal Basic Income, AI Safety, Coffee vs Tea")
            rounds_input = gr.Slider(minimum=1, maximum=10, value=3, step=1, label="Rounds")
            model_pro_input = gr.Dropdown(
                choices=["google/gemini-2.0-flash-001", "openai/gpt-3.5-turbo", "anthropic/claude-3-haiku"],
                value="google/gemini-2.0-flash-001",
                label="Pro Model"
            )
            model_con_input = gr.Dropdown(
                choices=["google/gemini-2.0-flash-001", "openai/gpt-3.5-turbo", "anthropic/claude-3-haiku"],
                value="google/gemini-2.0-flash-001",
                label="Con Model"
            )
            start_btn = gr.Button("Start Debate", variant="primary")
        
        with gr.Column(scale=4):
            # Labels for the chat bubbles
            chatbot = gr.Chatbot(label="Debate Transcript (User=Pro, Bot=Con)")

    start_btn.click(
        fn=run_debate,
        inputs=[topic_input, rounds_input, model_pro_input, model_con_input],
        outputs=[chatbot]
    )

if __name__ == "__main__":
    demo.launch()

/var/folders/6m/3v81d3yj54z2fqnfs4v_zxdr0000gn/T/ipykernel_9811/222845367.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/Users/mubaraq/Documents/Python/AI Projects/.venv/lib/python3.12/site-packages/gradio/utils.py:1187: UserWarning: Expected maximum 3 arguments for function <function run_debate at 0x11c4bcf40>, received 4.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
